## Create a 2D netCDF file 

A flexible notebook to convert data files into netCDF format for upload to Datalakes. This is for 2D data (e.g. temperature at various depths measured over time -- dimension1 = depth, dimension2 = time). 

If you are not sure how many dimensions your data has, please refer to the netCDF_README file.

This python notebook is designed to be flexible to handle various different types of data. Thus it requires user entry into a few files to provide information about the data. 

In [20]:
# import the needed libraries
import os
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib as plt
from pathlib import Path
import yaml
import csv
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
from collections import defaultdict

In [3]:
# import the functions that exist in the folder
from utils import find_file, extract_identifier_from_pattern, sanitize_name

### Step 1: what format is your data?

**NOTE:** We are assuming that one of your dimensions is time. The other dimension (we will call dimension1) could be a variety of different things; common ones on Datalakes are depth or water pressure.


Let's collect some information about your data files. 

* Does each file only contain one dimension1 value? (e.g. file_1 is measurements of variables at a specific depth, there can be multiple files with the same depth - ex. file_1m_datecollected, file_1m_datecollected2 ...). Then we want to use OPTION A. 

* If a single file has multiple dimension1 values (e.g. file_1 has temperature readings at different depths and depth is your dimension1), use OPTION B. See other script.

* If you have both options, then run the script twice and then you can easily combine your netCDF files. 

In [12]:
# Where is the dimension1 information? Choose ONE:
# Uncomment your choice

# OPTION A: a file does not contain multiple dimension1 values:
# one file = one dimension value (but can have multiple files with the same dimensional value) 
dim_source = 'filename'
# What is the variable name of dimension1? (e.g. depth, water_pressure, ...) do not include spaces
dimension1_name = 'depth'

### Step 2: Find all the data files

In [13]:
# Migrate all the data into the data folder

# the working directory is the folder that contains the full repo:
working_directory = Path.cwd().parent
# the input data directory is where all your data should be:
inputdata_directory = working_directory / 'data/input/L0'
# the script directory is where all the scripts are located:
script_directory = working_directory / 'scripts'

# What is the file naming convention (need help with this - see README.txt -> understanding components of the code, Step 1)
# if possible, the idvalue should be associated with the dimension value (e.g. the actual value, an instrument number, ...)
data_file_pattern = 'AUTO_{idvalue}_*_data.txt'

In [16]:
### User input not required in this cell, if naming in above cell has not changed ###

# find all the data files you have put in the data folder (L0):
file_list, file_summary = find_file(inputdata_directory, data_file_pattern)  

# create a metadata file about the data files
dfile_metadata = pd.DataFrame(list(file_summary['files_per_identifier'].items()), columns = ['ID_value', 'number_of_files'])
# add the dimension variable
dimension1_name = sanitize_name(dimension1_name) #remove any spaces, replace with underscores for ease of handling
dfile_metadata[dimension1_name] = ""

# save this info to a csv file
dfile_output_file = inputdata_directory.parent / 'datafile_information.csv'
dfile_metadata.to_csv(dfile_output_file, index = False)

Total files found: 57
Total number of unqiue 'idvalues' found: 19


### Step 3: edit the datafile_information file

In your data/input folder there is now a file labeled 'datafile_information.csv' which has a list of the unique id_values, the number of files per id_value (e.g. how many time steps), and a blank column with your dimension1_name. 

**Check the information in this file and enter the dimension1 values for each id_value.**

If there are id_values you do not wish to process, simply remove the row from this file. We will be using this file to process the data files. 

### Step 4: information about the format of your data files


In [26]:
# Data collection time period (YYYY,mm,dd,hh,min,ss):
start_time = datetime(2025,1,17,0,0,0)    
end_time =  datetime(2025,4,22,0,0,0)

# Do your data files have headers? How many lines of header/text? We will skip those to get to the data.
skiprows = 1

# What is the separator in your data file (e.g. comma = ',', tab = '\t', space = ' ', semicolon = ';', etc)?
separator = ','

# What are your date/time column(s) named in your data file? 
# Ordered from largest to smallest (e.g. Date, Time OR Year, Month, Day, etc.) NOT in the order they are in your datafile
time_columns = ['Time']

# Do your data files collect data in UTC time? Enter the time shift in hours or 0 if they are already in UTC:
time_shift = 0

# What are the variable names in the data files (include the names of the time/date columns)? 
# We will pull the variables in your file based on these names.
# Only include variables you are interested in processing:
var_names = ['Time', 'Temp']

In [29]:
### User input not required in this cell ###

# Convert start and end time to seconds since 1970 for netcdf 
sec_start_time = start_time.timestamp()
sec_end_time = end_time.timestamp() 

# Create a variable mapping file:
var_output = inputdata_directory.parent / 'variables.csv'

#Do not edit the below:
headers = ['var_name', 'dim', 'unit', 'long_name', 'min', 'max']

# add in dimension1 to this file
dim1_metadata = {'var_name': dimension1_name, 
                 'dim': f'({dimension1_name},)',
                 'unit': '',
                 'long_name': '',
                 'min': '',
                 'max': ''}

# input the time metadata that we already know
time_metadata = {'var_name':'time',
                 'dim': '(time,)',     #var is dependent on this variable
                 'unit': 'seconds since 1970-01-01 00:00:00',
                 'long_name': 'time',
                 'min': sec_start_time - 86400,
                 'max': sec_end_time + 86400}

# let's make the variable file
with open(var_output, mode = 'w', newline='') as file:
     writer = csv.DictWriter(file, fieldnames=headers)
     writer.writeheader()
    
     # write time 
     writer.writerow(time_metadata)
     # write the other dimension in
     writer.writerow(dim1_metadata)
    
     # manage the overlapping time entries if they exist
     list3 = [item for item in var_names if item not in time_columns]
    
     # write in the other variables
     for name in list3:
         #remove any spaces as needed
         name = sanitize_name(name)
         row = {
             'var_name': name.lower(),
             'dim':'',
             'unit': '',
             'long_name': '',
             'min':'',
             'max': ''}
         writer.writerow(row)

### Step 5: edit the variable file

In your data/input folder there is now a file labeled 'variables.csv'. This information is needed for the netCDF metadata. 

**Check the information in this file and fill in the blank columns for your additional variables. Do not edit the time row.**

Info about this file:
* var_name = shorthand name for the variable (ensure there are no spaces in the var_name)
* dim = the dimension(s) this is variable is dependent on, set up as (dim1, dim2, ...) (see README.md Step 5 for more information, if you are uncertain about this)
* unit = the units (e.g. temp = degrees C etc)
* long_name = the full name for the variable (can be the same as var_name)
* min = expected minimum value
* max = expected maximum value

### Step 6: read in all the data files

It's show time. 

In [41]:
# first we will cycle through all the files, based on the datafile_information.csv file
# open them, based on the information provided, then combine them into a netcdf 

